# Pipeline 2 — Limitar abstracts (frases mais relevantes por paper/categoria)

Entrada: `arxiv_amostra_1500.json` (saída da Pipeline 1, JSON Lines).

Objetivo: **reduzir o tamanho de cada abstract** mantendo apenas as **frases inteiras mais relevantes**, para não sobrecarregar a rede neural na geração de embeddings. Manter **frases completas** preserva o contexto e a ordem das palavras — o que o **BERT** precisa para gerar bons embeddings. **Títulos e categorias ficam intactos.**

**Como pontuamos as frases:** cada frase recebe a soma dos pesos TF-IDF das suas palavras, combinando dois sinais —
1. **Relevância no paper** (TF-IDF do documento): termos distintivos daquele abstract.
2. **Relevância na categoria** (TF-IDF por categoria): termos característicos da área (`assigned_category`).

`score(frase) = Σ (tfidf_doc + LAMBDA_CAT * tfidf_categoria)` sobre as palavras da frase. Mantemos as **top-N frases** e as devolvemos na **ordem original** do abstract (texto continua coeso). Stopwords em inglês não contam para o score.

**Resultado:** `arxiv_amostra_1500_abstracts_limitados.json` (JSON Lines), com o campo novo `abstract_reduzido`.

## 1. Configuração

In [1]:
N_FRASES   = 3        # nº de frases mantidas por abstract
LAMBDA_CAT = 0.5      # peso do sinal de categoria (0 = só paper; 1 = paper+categoria iguais)

# token: palavras de letras (e hífen), com 3+ caracteres
TOKEN_RE = r"(?u)\b[a-zA-Z][a-zA-Z-]{2,}\b"

print(f"Limite: {N_FRASES} frases/abstract | peso categoria = {LAMBDA_CAT}")

Limite: 3 frases/abstract | peso categoria = 0.5


## 2. Montar o Drive e carregar a amostra (JSON Lines)

In [2]:
from google.colab import drive
import os, pandas as pd
drive.mount('/content/drive')

BASE       = '/content/drive/MyDrive/projetoIA-EquipeLoremIpsum'
CAMINHO_IN = os.path.join(BASE, 'pipelines', 'arxiv_amostra_1500.json')
SAIDA      = os.path.join(BASE, 'pipelines', 'arxiv_amostra_1500_abstracts_limitados.json')

df = pd.read_json(CAMINHO_IN, lines=True)
print('Carregado:', df.shape)
print('Colunas:', list(df.columns))
df[['id', 'title', 'assigned_category']].head()

Mounted at /content/drive
Carregado: (1500, 8)
Colunas: ['id', 'title', 'abstract', 'categories', 'primary_category', 'assigned_category', 'authors', 'update_date']


,id,title,assigned_category
0,704.0671,Learning from compressed observations,cs.LG
1,704.0954,Sensor Networks with Random Links: Topology De...,cs.LG
2,704.1020,The on-line shortest path problem under partia...,cs.LG
3,704.1028,A neural network approach to ordinal regression,cs.LG
4,704.1274,Parametric Learning and Monte Carlo Optimization,cs.LG


## 3. TF-IDF — nível documento e nível categoria

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# (1) TF-IDF por documento (cada abstract é um documento)
vec_doc = TfidfVectorizer(stop_words='english', token_pattern=TOKEN_RE, lowercase=True)
X_doc   = vec_doc.fit_transform(df['abstract']).tocsr()
vocab_d = vec_doc.get_feature_names_out()

# (2) TF-IDF por categoria (todos os abstracts da categoria concatenados = 1 documento)
cat_text = df.groupby('assigned_category')['abstract'].apply(' '.join)
vec_cat  = TfidfVectorizer(stop_words='english', token_pattern=TOKEN_RE, lowercase=True)
X_cat    = vec_cat.fit_transform(cat_text.values).tocsr()
vocab_c  = vec_cat.get_feature_names_out()

# pré-calcula os scores de categoria como dicionários {palavra: peso}
cat_pos    = {c: i for i, c in enumerate(cat_text.index)}
cat_scores = {}
for c, i in cat_pos.items():
    row = X_cat.getrow(i)
    cat_scores[c] = {vocab_c[j]: v for j, v in zip(row.indices, row.data)}

print('Vocabulário (documentos):', len(vocab_d))
print('Vocabulário (categorias):', len(vocab_c))

Vocabulário (documentos): 12645
Vocabulário (categorias): 12645


## 4. Reduzir os abstracts (extração das N frases mais relevantes)

In [4]:
import re
import nltk

# divisor de frases do NLTK
for pkg in ['punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{pkg}')
    except LookupError:
        nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize

# --- edge case: abreviações com ponto no meio que o NLTK divide errado ---
# protegemos o ponto com um sentinela (U+2024, ONE DOT LEADER) antes de dividir
# e restauramos depois. Ex.: "i.i.d." não vira fim de frase.
_ABREV = ['i.i.d.', 'e.g.', 'i.e.', 'et al.', 'al.', 'etc.', 'vs.', 'cf.', 'resp.',
          'w.r.t.', 'a.k.a.', 'Fig.', 'Eq.', 'Eqs.', 'Ref.', 'Refs.', 'approx.',
          'Dr.', 'Prof.', 'No.', 'Inc.', 'Sec.', 'Thm.', 'Eq.', 'Sr.', 'St.']
_SENT = '․'   # sentinela que substitui o '.' temporariamente

def _proteger(t):
    for a in _ABREV:
        t = t.replace(a, a.replace('.', _SENT))
    return t

def dividir_frases(texto):
    texto = ' '.join(texto.split())                 # normaliza quebras de linha/espaços
    texto = _proteger(texto)                         # blinda as abreviações
    try:
        frases = sent_tokenize(texto)
    except Exception:                                # fallback: split simples por pontuação
        frases = re.split(r'(?<=[.!?])\s+(?=[A-Z])', texto)
    frases = [f.replace(_SENT, '.').strip() for f in frases]   # restaura os pontos
    return [f for f in frases if f]

def score_frase(frase, doc_s, cat_s):
    # palavras únicas da frase (evita que repetição infle o score)
    palavras = {w.lower() for w in re.findall(TOKEN_RE, frase)}
    return sum(doc_s.get(w, 0.0) + LAMBDA_CAT * cat_s.get(w, 0.0) for w in palavras)

def reduzir_abstract(i, abstract, categoria):
    """Mantém as top-N frases mais relevantes (paper + categoria), na ordem original."""
    row   = X_doc.getrow(i)
    doc_s = {vocab_d[j]: v for j, v in zip(row.indices, row.data)}
    cat_s = cat_scores.get(categoria, {})

    frases = dividir_frases(abstract)
    if len(frases) <= N_FRASES:
        return ' '.join(frases)                      # já é curto: mantém tudo

    pont = [(k, score_frase(f, doc_s, cat_s)) for k, f in enumerate(frases)]
    top  = sorted(pont, key=lambda x: x[1], reverse=True)[:N_FRASES]
    idx  = sorted(k for k, _ in top)                 # volta à ordem original do abstract
    return ' '.join(frases[k] for k in idx)

df['abstract_reduzido'] = [
    reduzir_abstract(i, ab, cat)
    for i, (ab, cat) in enumerate(zip(df['abstract'], df['assigned_category']))
]

# estatísticas de redução
df['n_palavras_orig']     = df['abstract'].str.split().str.len()
df['n_palavras_reduzido'] = df['abstract_reduzido'].str.split().str.len()
df['n_frases_orig']       = df['abstract'].map(lambda t: len(dividir_frases(t)))
df['n_frases_reduzido']   = df['abstract_reduzido'].map(lambda t: len(dividir_frases(t)))
print('Frases  (orig)   média:', round(df['n_frases_orig'].mean(), 1))
print('Palavras(orig)   média:', round(df['n_palavras_orig'].mean(), 1))
print('Palavras(reduzido) média:', round(df['n_palavras_reduzido'].mean(), 1))
print('Redução média: {:.0%}'.format(1 - df['n_palavras_reduzido'].sum() / df['n_palavras_orig'].sum()))

Frases  (orig)   média: 5.1
Palavras(orig)   média: 119.2
Palavras(reduzido) média: 79.4
Redução média: 33%


In [5]:
# Exemplo antes/depois
ex = df.iloc[0]
print('CATEGORIA:', ex['assigned_category'])
print('TÍTULO   :', ex['title'])
print('\n--- ORIGINAL ---\n', ex['abstract'])
print('\n--- REDUZIDO ---\n', ex['abstract_reduzido'])

CATEGORIA: cs.LG
TÍTULO   : Learning from compressed observations

--- ORIGINAL ---
 The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illust

## 5. Salvar como JSON Lines e baixar para o PC

Mesma lógica da Pipeline 1: salva no Drive (pasta `pipelines`) e baixa para a sua máquina. Mantém todos os campos originais (título e categorias **intactos**) e adiciona `abstract_reduzido` (as N frases mais relevantes). Use o campo `abstract_reduzido` como entrada do BERT na Pipeline 3.

In [6]:
df.to_json(SAIDA, orient='records', lines=True, force_ascii=False)
print('Salvo:', SAIDA, '|', len(df), 'registros (JSON Lines)')

try:
    from google.colab import files
    files.download(SAIDA)
except Exception as e:
    print('Download automático indisponível (rodando fora do Colab?):', e)

Salvo: /content/drive/MyDrive/projetoIA-EquipeLoremIpsum/pipelines/arxiv_amostra_1500_abstracts_limitados.json | 1500 registros (JSON Lines)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>